In [1]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator


def load_event_accumulator(path: str | Path) -> EventAccumulator:
    ea = EventAccumulator(str(path), size_guidance={"scalars": 0})
    ea.Reload()
    return ea


def load_all_scalars(path: str | Path) -> dict[str, pd.DataFrame]:
    """
    Load all scalar tags from one TensorBoard logdir or event file.

    Returns:
        {
            tag_name: DataFrame(step, value)
        }
    """
    ea = load_event_accumulator(path)
    tags = ea.Tags().get("scalars", [])

    out: dict[str, pd.DataFrame] = {}
    for tag in tags:
        events = ea.Scalars(tag)
        out[tag] = pd.DataFrame(
            {
                "step": [e.step for e in events],
                "value": [e.value for e in events],
            }
        )
    return out


def _get_selected_row(
        tag_map: dict[str, pd.DataFrame],
        reference_tag: str,
        mode: str,
) -> tuple[int, int]:
    """
    Returns:
        selected_idx, selected_step
    """
    if reference_tag not in tag_map:
        raise KeyError(f"Reference tag '{reference_tag}' not found.")

    ref_df = tag_map[reference_tag].sort_values("step").reset_index(drop=True)
    if ref_df.empty:
        raise ValueError(f"Reference tag '{reference_tag}' is empty.")

    if mode == "max":
        selected_idx = int(ref_df["value"].idxmax())
    elif mode == "min":
        selected_idx = int(ref_df["value"].idxmin())
    else:
        raise ValueError("mode must be either 'max' or 'min'")

    selected_step = int(ref_df.loc[selected_idx, "step"])
    return selected_idx, selected_step


def compare_ablation_runs(
        baseline_paths: list[str | Path],
        ablated_paths: list[str | Path],
        reference_tag: str,
        mode: str = "max",
        seed_labels: list[int | str] | None = None,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Compare baseline vs ablated runs using each run's own best checkpoint
    selected from `reference_tag`.

    For each paired run:
      - baseline best index is selected independently on baseline run
      - ablated best index is selected independently on ablated run
      - all tags are read at those selected indices
      - per-tag delta = ablated_value - baseline_value

    Returns:
      per_seed_df:
        one row per (seed, tag)

      summary_df:
        one row per tag with baseline mean, ablated mean, delta mean/std
    """
    if len(baseline_paths) != len(ablated_paths):
        raise ValueError("baseline_paths and ablated_paths must have the same length")

    if seed_labels is None:
        seed_labels = list(range(len(baseline_paths)))

    if len(seed_labels) != len(baseline_paths):
        raise ValueError("seed_labels must have the same length as the path lists")

    rows = []

    for seed, baseline_path, ablated_path in zip(seed_labels, baseline_paths, ablated_paths):
        baseline_tags = load_all_scalars(baseline_path)
        ablated_tags = load_all_scalars(ablated_path)

        b_idx, b_step = _get_selected_row(baseline_tags, reference_tag, mode)
        a_idx, a_step = _get_selected_row(ablated_tags, reference_tag, mode)

        common_tags = baseline_tags.keys() & ablated_tags.keys()

        for tag in common_tags:
            b_df = baseline_tags[tag].sort_values("step").reset_index(drop=True)
            a_df = ablated_tags[tag].sort_values("step").reset_index(drop=True)

            if b_idx >= len(b_df) or a_idx >= len(a_df):
                continue

            b_row = b_df.iloc[b_idx]
            a_row = a_df.iloc[a_idx]

            b_val = float(b_row["value"])
            a_val = float(a_row["value"])

            rows.append(
                {
                    "seed": seed,
                    "tag": tag,
                    "baseline_idx": b_idx,
                    "ablated_idx": a_idx,
                    "baseline_step": int(b_row["step"]),
                    "ablated_step": int(a_row["step"]),
                    "baseline_value": b_val,
                    "ablated_value": a_val,
                    "delta": a_val - b_val,
                }
            )

    per_seed_df = pd.DataFrame(rows)
    if per_seed_df.empty:
        return per_seed_df, per_seed_df

    summary_df = (
        per_seed_df.groupby("tag", as_index=False)
        .agg(
            baseline_mean=("baseline_value", "mean"),
            baseline_std=("baseline_value", "std"),
            ablated_mean=("ablated_value", "mean"),
            ablated_std=("ablated_value", "std"),
            delta_mean=("delta", "mean"),
            delta_std=("delta", "std"),
            n=("delta", "count"),
            baseline_min_step=("baseline_step", "min"),
            baseline_max_step=("baseline_step", "max"),
            ablated_min_step=("ablated_step", "min"),
            ablated_max_step=("ablated_step", "max"),
        )
        .sort_values("tag")
        .reset_index(drop=True)
    )

    return per_seed_df, summary_df

To evaluate the good contribution of modalities we see MRR of it with and without the metric <br>
So if I have EEG, Aud, Txt and Vid and decide to ablate *vid* I measure:

A:MRR_mean modalities model that can use Vid but without video in inputstream <br>
B:MRR_mean across modalities of the ablated model

If B > A → Video is likely hurting other modalities<br>
If B < A → Video is likely helping them<br>
If B ~ A → Video has little effect on them<br>

Text seems to fluctuate a lot from seed to seed. <br>
The modality is unstable.

> What if the increase in performance by removing audio is because it mitigates the presence of txt and thus removing modaltiies while txt is in it always proves gains?

# MoCo less

In [2]:
from lightning.pytorch.loggers import TensorBoardLogger
from hydra import compose, initialize
import lightning
from main.model.neegavi.train_utils import KdTrainDataModule
from main.model.neegavi.training import EasyEegAviKdVateMaskedModule

from main.model.script.hydra_beans import KdConfig
from main.model.neegavi.factories.core import CoreFactory
from main.model.neegavi.utils import get_model_ckpt

config_name = "train-local.yaml"
with initialize(version_base=None, config_path="../../../conf/"):
    cfg: KdConfig = compose(config_name="train-local.yaml")

trainer = lightning.Trainer(precision="16-mixed",
                            logger=TensorBoardLogger("tb_logs", name="moco-less-valid-mod-ablate", version="1"))

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

/home/jacopo/PycharmProjects/progetto-tesi/.venv/lib/python3.12/site-packages/hydra/_internal/defaults_list.py:251: UserWarning: In 'train-local.yaml': Defaults list is missing `_self_`. See https://hydra.cc/docs/1.2/upgrades/1.0_to_1.1/default_composition_order for more information
  warnings.warn(msg, UserWarning)
Using 16bit Automatic Mixed Precision (AMP)
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


Baseline model load

In [3]:
baselines = []
seed_ckpt = [
    # Seed=1
    #"/home/jacopo/PycharmProjects/progetto-tesi/main/model/script/outputs/ablation-moco-best/2026-03-26_11-25-28/checkpoints/epochepoch=36-stepstep=94461.ckpt",
    # Seed=42. For some reason this are broken. (Model architecture mismatch on keys?)
    "/home/jacopo/PycharmProjects/progetto-tesi/main/model/script/outputs/ablation-moco-default/2026-04-02_11-31-29/checkpoints/epochepoch=37-stepstep=97014.ckpt",
    # Seed=150
    #"/home/jacopo/PycharmProjects/progetto-tesi/main/model/script/outputs/ablation-moco-best/2026-03-24_00-32-18/checkpoints/epochepoch=38-stepstep=99567.ckpt",
]

for path in seed_ckpt:
    ckpt = get_model_ckpt(path)
    baseline = CoreFactory.best_inference(disabled_supports={}).build()
    # Load state of the seed ckpt
    baseline.load_state_dict(ckpt, strict=False)
    baseline.eval()
    # Append the built model
    baselines.append(baseline)

# Text

In [ ]:
mrr_check_keys = [
    'val/fused/mrr_ecg',
    'val/fused/mrr_aud',
    'val/fused/mrr_vid',
    # 'val/fused/mrr_txt', -> We are ablating text
    'val/fused/mrr_eeg',
    'val/fused/mrr_mean',
]

In [ ]:
txt_ablate_ckpt_150 = "/home/jacopo/PycharmProjects/progetto-tesi/main/model/script/outputs/ablation-txt/2026-03-25_12-03-51/checkpoints/epochepoch=38-stepstep=99567.ckpt"

In [ ]:
# Create model instance
inference_ckpt = get_model_ckpt(txt_ablate_ckpt_150)
mod_less_model = CoreFactory.best_inference(disabled_supports={'txt'}).build()
mod_less_model.load_state_dict(inference_ckpt, strict=False)
mod_less_model.eval()

# And its according datamodule
datamodule = KdTrainDataModule(
    dataset_paths=cfg.dataset_descriptors,
    batch_size=cfg.trainer.batch_size,
    seed=cfg.data_seed,
    dequantize_keys=["eeg", "aud", "vid", "txt", "ecg"],
    # All except currently ablated modality
    take_keys=[mod_less_model.pivot.code] + mod_less_model.fusion_keys()
)

# Now validate it
mod_less_module = EasyEegAviKdVateMaskedModule(mod_less_model, None, datamodule=datamodule, )
mod_less_results = trainer.validate(mod_less_module, datamodule=datamodule)

# I only care for MRR at the moment
print("For seed=150")
for key in mrr_check_keys:
    print("ablated_res for key:", key, " is:", mod_less_results[0][key])
    print("key:", key, " has gain of:", mod_less_results[0][key] - baseline_metrics[key], "\n")

## Baseline without mod

In [ ]:
baseline_mod_less_results = []
for b in baselines[1:]:
    baseline_mod_less_module = EasyEegAviKdVateMaskedModule(b, None, datamodule=datamodule, )
    baseline_mod_less_results.append(trainer.validate(baseline_mod_less_module, datamodule=datamodule)[0])

baseline_mod_less_metrics = {}
for key in mrr_check_keys:
    values = []
    for res in baseline_mod_less_results:
        values.append(res[key])

    arr = np.array(values)
    baseline_mod_less_metrics[key] = np.mean(arr)
    print(f"For key:{key} on {len(baseline_mod_less_results)} baselines mean: {np.mean(arr)} std: {np.std(arr)}")

# Compare now
print("For seed=150")
for key in mrr_check_keys:
    print("ablated_res for key:", key, " is:", mod_less_results[0][key])
    print("key:", key, " has gain of:", mod_less_results[0][key] - baseline_mod_less_metrics[key], "\n")

> The model appears robust to the absence of text at inference. Training without text still yields a modest improvement on shared-modality retrieval, suggesting that text may slightly complicate alignment during training rather than being strictly required at test time.

> Although removing text slightly improved shared-modality retrieval, the gain was modest. Since the broader goal of the model is to learn richer multimodal representations, text was retained as a support modality despite this small alignment cost.

or

> In this setting, text was derived from speech rather than being an independent source of information. Given its small negative effect on shared-modality retrieval and the likely redundancy with audio, it was excluded from the final configuration.

# Audio

In [12]:
base_path = "/home/jacopo/PycharmProjects/progetto-tesi/main/model/script/outputs/"

In [13]:
# 1, 42, 150
baseline_paths = [
    base_path + "ablation-moco-best/2026-03-26_11-25-28/tb_logs/EEGAVI-1/0/events.out.tfevents.1774520728.jacopo-MS-7C91.131834.0",
    base_path + "ablation-moco-default/2026-04-02_11-31-29/tb_logs/EEGAVI-42/0/events.out.tfevents.1775122290.jacopo-MS-7C91.673941.0",
    base_path + "ablation-moco-best/2026-03-24_00-32-18/tb_logs/EEGAVI-150/0/events.out.tfevents.1774308739.jacopo-MS-7C91.313312.0",
]

ablated_paths = [
    base_path + "ABLATION-FINAL-noMoCo-aud/2026-04-01_12-02-08/tb_logs/EEGAVI-1/0/events.out.tfevents.1775037728.jacopo-MS-7C91.604042.0",
    base_path + "ABLATION-FINAL-noMoCo-aud/2026-03-31_21-14-07/tb_logs/EEGAVI-42/0/events.out.tfevents.1774984447.jacopo-MS-7C91.550376.0",
    base_path + "ablation-aud/2026-03-25_22-37-18/tb_logs/EEGAVI-150/0/events.out.tfevents.1774474638.jacopo-MS-7C91.107113.0",
]

per_seed_df, summary_df = compare_ablation_runs(
    baseline_paths=baseline_paths,
    ablated_paths=ablated_paths,
    reference_tag="val/fused/mrr_mean",
    mode="max",
    seed_labels=[1, 42, 150],
)

In [ ]:
per_seed_df[per_seed_df["tag"] == "val/fused/mrr_mean"]

In [ ]:
summary_df[summary_df["tag"] == "val/fused/mrr_mean"]

In [ ]:
shared_tags = ["val/fused/mrr_vid", "val/fused/mrr_txt", "val/fused/mrr_ecg", "val/fused/mrr_eeg", ]

mrr_shared = (
    per_seed_df[per_seed_df["tag"].isin(shared_tags)].groupby("seed", as_index=False)
    .agg(baseline_value=("baseline_value", "mean"), ablated_value=("ablated_value", "mean"), )
)

mrr_shared["delta"] = mrr_shared["baseline_value"] - mrr_shared["ablated_value"]
print(mrr_shared)

corrected_summary = {
    "baseline_mean": mrr_shared["baseline_value"].mean(),
    "baseline_std": mrr_shared["baseline_value"].std(ddof=1),
    "ablated_mean": mrr_shared["ablated_value"].mean(),
    "ablated_std": mrr_shared["ablated_value"].std(ddof=1),
    "delta_mean": mrr_shared["delta"].mean(),
    "delta_std": mrr_shared["delta"].std(ddof=1),
}
print("Summary:")
corrected_summary

In [ ]:
import torch

t0 = torch.tensor([0.722, 0.739, 0.736])

In [ ]:
t1 = torch.tensor([0.743, 0.751, 0.752])
t = t0 - t1


In [ ]:
t.mean()

In [ ]:
t.std()

## Baseline without mod

In [ ]:
datamodule = KdTrainDataModule(
    dataset_paths=cfg.dataset_descriptors,
    batch_size=cfg.trainer.batch_size,
    seed=cfg.data_seed,
    dequantize_keys=["eeg", "aud", "vid", "txt", "ecg"],
    # All except currently ablated modality
    take_keys=["eeg", "vid", "txt", 'aud' "ecg"],
)

baseline_mod_less_results = []
for b in baselines:
    baseline_mod_less_module = EasyEegAviKdVateMaskedModule(b, None, datamodule=datamodule, )
    baseline_mod_less_results.append(trainer.validate(baseline_mod_less_module, datamodule=datamodule)[0])

In [ ]:
baseline_mod_less_results[0]["val/fused/mrr_aud"]

In [ ]:
baseline_mod_less_paths = [
    "/home/jacopo/PycharmProjects/progetto-tesi/main/model/ablation/tb_logs/moco-less-valid-mod-ablate/1/events.out.tfevents.1775118478.jacopo-MS-7C91.670271.1",
    "/home/jacopo/PycharmProjects/progetto-tesi/main/model/ablation/tb_logs/moco-less-valid-mod-ablate/1/events.out.tfevents.1775118517.jacopo-MS-7C91.670271.2",
    "/home/jacopo/PycharmProjects/progetto-tesi/main/model/ablation/tb_logs/moco-less-valid-mod-ablate/1/events.out.tfevents.1775118558.jacopo-MS-7C91.670271.3",
]

per_seed_df, summary_df = compare_ablation_runs(
    baseline_paths=[baseline_mod_less_paths[0], baseline_mod_less_paths[2]],
    ablated_paths=[ablated_paths[0], baseline_mod_less_paths[2]],
    reference_tag="val/fused/mrr_mean",
    mode="max",
    seed_labels=[1, 150],
)
# TODO Ho sbaglaito, mean va senza fare su ablated modality in baseline cazzo

In [ ]:
per_seed_df[per_seed_df["tag"] == "val/fused/mrr_mean"]

In [ ]:
shared_tags = [
    "val/fused/mrr_vid",
    "val/fused/mrr_txt",
    "val/fused/mrr_ecg",
    "val/fused/mrr_eeg",
]

mrr_noaud_shared = (
    per_seed_df[per_seed_df["tag"].isin(shared_tags)]
    .groupby("seed", as_index=False)
    .agg(
        baseline_value=("baseline_value", "mean"),
        ablated_value=("ablated_value", "mean"),
    )
)

mrr_noaud_shared["delta"] = (
        mrr_noaud_shared["baseline_value"] - mrr_noaud_shared["ablated_value"]
)

print(mrr_noaud_shared)

In [ ]:
per_seed_df[per_seed_df["tag"] == "val/fused/mrr_vid"]
per_seed_df[per_seed_df["tag"] == "val/fused/mrr_txt"]
per_seed_df[per_seed_df["tag"] == "val/fused/mrr_ecg"]
per_seed_df[per_seed_df["tag"] == "val/fused/mrr_eeg"]

In [ ]:
summary_df[summary_df["tag"] == "val/fused/mrr_mean"]

# ECG

In [14]:
base_path = "/home/jacopo/PycharmProjects/progetto-tesi/main/model/script/outputs/"
# 1, 42, 150
baseline_paths = [
    base_path + "ablation-moco-best/2026-03-26_11-25-28/tb_logs/EEGAVI-1/0/events.out.tfevents.1774520728.jacopo-MS-7C91.131834.0",
    base_path + "ablation-moco-best/2026-03-19_15-32-23/tb_logs/TODO/0/events.out.tfevents.1773930743.jacopo-MS-7C91.20883.0",
    base_path + "ablation-moco-best/2026-03-24_00-32-18/tb_logs/EEGAVI-150/0/events.out.tfevents.1774308739.jacopo-MS-7C91.313312.0",
]

ablated_paths = [
    base_path + "ablation-ecg/2026-03-26_19-59-53/tb_logs/EEGAVI-1/0/events.out.tfevents.1774551593.jacopo-MS-7C91.157387.0",
    base_path + "ABLATION-FINAL-noMoCo-ecg/2026-04-01_17-32-54/tb_logs/EEGAVI-42/0/events.out.tfevents.1775057574.jacopo-MS-7C91.627600.0",
    base_path + "ablation-ecg/2026-03-26_03-59-29/tb_logs/EEGAVI-150/0/events.out.tfevents.1774493970.jacopo-MS-7C91.121009.0",
]

per_seed_df, summary_df = compare_ablation_runs(
    baseline_paths=baseline_paths,
    ablated_paths=ablated_paths,
    reference_tag="val/fused/mrr_mean",
    mode="max",
    seed_labels=[1, 42, 150],
)

In [ ]:
per_seed_df[per_seed_df["tag"] == "val/fused/mrr_mean"]

In [ ]:
summary_df[summary_df["tag"] == "val/fused/mrr_mean"]

In [ ]:
shared_tags = ["val/fused/mrr_vid", "val/fused/mrr_txt", "val/fused/mrr_aud", "val/fused/mrr_eeg", ]

mrr_shared = (
    per_seed_df[per_seed_df["tag"].isin(shared_tags)].groupby("seed", as_index=False)
    .agg(baseline_value=("baseline_value", "mean"), ablated_value=("ablated_value", "mean"), )
)

mrr_shared["delta"] = mrr_shared["baseline_value"] - mrr_shared["ablated_value"]
print(mrr_shared)

corrected_summary = {
    "baseline_mean": mrr_shared["baseline_value"].mean(),
    "baseline_std": mrr_shared["baseline_value"].std(ddof=1),
    "ablated_mean": mrr_shared["ablated_value"].mean(),
    "ablated_std": mrr_shared["ablated_value"].std(ddof=1),
    "delta_mean": mrr_shared["delta"].mean(),
    "delta_std": mrr_shared["delta"].std(ddof=1),
}
print("Summary:")
corrected_summary

In [ ]:
ecg_ablate_ckpt_150 = "/home/jacopo/PycharmProjects/progetto-tesi/main/model/script/outputs/ablation-ecg/2026-03-26_03-59-29/checkpoints/epochepoch=39-stepstep=102120.ckpt"

In [ ]:
# Create model instance
inference_ckpt = get_model_ckpt(ecg_ablate_ckpt_150)
mod_less_model = CoreFactory.best_inference(disabled_supports={'ecg'}).build()
mod_less_model.load_state_dict(inference_ckpt, strict=False)
mod_less_model.eval()
# todo 3 seeds
# And its according datamodule
datamodule = KdTrainDataModule(
    dataset_paths=cfg.dataset_descriptors,
    batch_size=cfg.trainer.batch_size,
    seed=cfg.data_seed,
    dequantize_keys=["eeg", "aud", "vid", "txt", "ecg"],
    # All except currently ablated modality
    take_keys=[mod_less_model.pivot.code] + mod_less_model.fusion_keys()
)

# Now validate it
mod_less_module = EasyEegAviKdVateMaskedModule(mod_less_model, None, datamodule=datamodule, )
mod_less_results = trainer.validate(mod_less_module, datamodule=datamodule)

# I only care for MRR at the moment
print("For seed=150")
for key in mrr_check_keys:
    print("ablated_res for key:", key, " is:", mod_less_results[0][key])
    print("key:", key, " has gain of:", mod_less_results[0][key] - baseline_metrics[key], "\n")

## Baseline without mod

In [7]:
datamodule = KdTrainDataModule(
    dataset_paths=cfg.dataset_descriptors,
    batch_size=cfg.trainer.batch_size,
    seed=cfg.data_seed,
    dequantize_keys=["eeg", "aud", "vid", "txt", "ecg"],
    # All except currently ablated modality
    take_keys=["eeg", "vid", "txt", "aud"],
)

baseline_mod_less_results = []
for b in baselines:
    baseline_mod_less_module = EasyEegAviKdVateMaskedModule(b, None, datamodule=datamodule, )
    baseline_mod_less_results.append(trainer.validate(baseline_mod_less_module, datamodule=datamodule)[0])

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Validation: |          | 0/? [00:00<?, ?it/s]

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃          Validate metric          ┃           DataLoader 0            ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│      val/fused/alignment_aud      │        0.14348331093788147        │
│      val/fused/alignment_eeg      │        0.1459227055311203         │
│      val/fused/alignment_txt      │        0.07862619310617447        │
│      val/fused/alignment_vid      │        0.16075308620929718        │
│       val/fused/margin_aud        │        0.3082556426525116         │
│       val/fused/margin_eeg        │        0.2949964106082916         │
│       val/fused/margin_txt        │        0.31494519114494324        │
│       val/fused/margin_vid        │        0.3892878592014313         │
│   val/fused/meanR@1-3-5-10_aud    │         0.970386803150177         │
│   val/fused/meanR@1-3-5-10_eeg    │        0.9715876579284668         │
│   val/fused/meanR@1-3-5-10_mean   │        0.8231160640716553         │
│   val/fused/meanR@1-3-5-10_txt    │        0.5324558615684509         │
│   val/fused/meanR@1-3-5-10_vid    │        0.8180338740348816         │
│         val/fused/mrr_aud         │        0.9523364305496216         │
│         val/fused/mrr_eeg         │        0.9597161412239075         │
│        val/fused/mrr_mean         │        0.7876248955726624         │
│         val/fused/mrr_txt         │        0.5057629942893982         │
│         val/fused/mrr_vid         │        0.7326840162277222         │
│        val/fused/top10_aud        │        0.9952802062034607         │
│        val/fused/top10_eeg        │        0.9894814491271973         │
│       val/fused/top10_mean        │        0.8758527040481567         │
│        val/fused/top10_txt        │        0.5792701244354248         │
│        val/fused/top10_vid        │        0.9393789172172546         │
│        val/fused/top1_aud         │        0.9257004261016846         │
│        val/fused/top1_eeg         │         0.941983699798584         │
│        val/fused/top1_mean        │         0.734183669090271         │
│        val/fused/top1_txt         │        0.45707449316978455        │
│        val/fused/top1_vid         │         0.611975908279419         │
│        val/fused/top3_aud         │        0.9748782515525818         │
│        val/fused/top3_eeg         │        0.9731284379959106         │
│        val/fused/top3_mean        │         0.82813960313797          │
│        val/fused/top3_txt         │        0.5369428396224976         │
│        val/fused/top3_vid         │        0.8276088833808899         │
│        val/fused/top5_aud         │        0.9856882095336914         │
│        val/fused/top5_eeg         │         0.981756865978241         │
│        val/fused/top5_mean        │        0.8542882204055786         │
│        val/fused/top5_txt         │        0.5565360188484192         │
│        val/fused/top5_vid         │        0.8931718468666077         │
│          val/fusion-loss          │        2.9611523151397705         │
│          val/fusion/aud           │        2.8832194805145264         │
│          val/fusion/eeg           │        2.8503901958465576         │
│          val/fusion/txt           │        3.5602474212646484         │
│          val/fusion/vid           │        2.7662570476531982         │
│             val/loss              │        2.9611523151397705         │
│    val/prefusion/alignment_aud    │        0.0961453914642334         │
│    val/prefusion/alignment_txt    │        0.11981650441884995        │
│    val/prefusion/alignment_vid    │        0.13856735825538635        │
│     val/prefusion/margin_aud      │       0.003876183647662401        │
│     val/prefusion/margin_txt      │       0.005871460773050785        │
│     val/prefusion/margin_vid      │      0.00012896626139990985       │
│ val/prefusion/meanR@1-3-5-10_aud  │        0.00137028016615659        │
│ va

In [10]:
import torch
torch.tensor([0.648, 0.634, 0.711]).std()

tensor(0.0410)

In [6]:
baseline_mod_less_results[0]

{'val/fusion/eeg': 2.9098658561706543,
 'val/fusion/vid': 2.890000343322754,
 'val/fusion/aud': 2.983867645263672,
 'val/fusion/txt': 3.5420920848846436,
 'val/fusion/ecg': 2.161313056945801,
 'val/fusion-loss': 2.945873498916626,
 'val/loss': 2.945873498916626,
 'val/fused/top1_eeg': 0.9359848499298096,
 'val/fused/top3_eeg': 0.9700878858566284,
 'val/fused/top5_eeg': 0.9801955223083496,
 'val/fused/top10_eeg': 0.9888240098953247,
 'val/fused/meanR@1-3-5-10_eeg': 0.9687730669975281,
 'val/fused/mrr_eeg': 0.9555048942565918,
 'val/fused/alignment_eeg': 0.1416132003068924,
 'val/fused/margin_eeg': 0.2861286401748657,
 'val/fused/top1_vid': 0.5685693025588989,
 'val/fused/top3_vid': 0.7824726104736328,
 'val/fused/top5_vid': 0.8539658784866333,
 'val/fused/top10_vid': 0.9097273349761963,
 'val/fused/meanR@1-3-5-10_vid': 0.7786837816238403,
 'val/fused/mrr_vid': 0.6916446089744568,
 'val/fused/alignment_vid': 0.15162013471126556,
 'val/fused/margin_vid': 0.36526721715927124,
 'val/prefusi

In [18]:
# Need more ECG and Aud seed, still in noise atm
baseline_mod_less_paths = [
    "/home/jacopo/PycharmProjects/progetto-tesi/main/model/ablation/tb_logs/moco-less-valid-mod-ablate/1/events.out.tfevents.1775121485.jacopo-MS-7C91.673578.0",
    "/home/jacopo/PycharmProjects/progetto-tesi/main/model/ablation/tb_logs/moco-less-valid-mod-ablate/1/events.out.tfevents.1775152448.jacopo-MS-7C91.684437.1",
    "/home/jacopo/PycharmProjects/progetto-tesi/main/model/ablation/tb_logs/moco-less-valid-mod-ablate/1/events.out.tfevents.1775121595.jacopo-MS-7C91.673578.2",
]

per_seed_df, summary_df = compare_ablation_runs(
    baseline_paths=baseline_mod_less_paths,
    ablated_paths=ablated_paths,
    reference_tag="val/fused/mrr_mean",
    mode="max",
    seed_labels=[1, 42, 150],
)
# TODO Ho sbaglaito, mean va senza fare su ablated modality in baseline cazzo

In [19]:
shared_tags = ["val/fused/mrr_vid", "val/fused/mrr_txt", "val/fused/mrr_aud", "val/fused/mrr_eeg", ]

mrr_shared = (
    per_seed_df[per_seed_df["tag"].isin(shared_tags)].groupby("seed", as_index=False)
    .agg(baseline_value=("baseline_value", "mean"), ablated_value=("ablated_value", "mean"), )
)

mrr_shared["delta"] = mrr_shared["baseline_value"] - mrr_shared["ablated_value"]
print(mrr_shared)

corrected_summary = {
    "baseline_mean": mrr_shared["baseline_value"].mean(),
    "baseline_std": mrr_shared["baseline_value"].std(ddof=1),
    "ablated_mean": mrr_shared["ablated_value"].mean(),
    "ablated_std": mrr_shared["ablated_value"].std(ddof=1),
    "delta_mean": mrr_shared["delta"].mean(),
    "delta_std": mrr_shared["delta"].std(ddof=1),
}
print("Summary:")
corrected_summary

   seed  baseline_value  ablated_value     delta
0     1        0.776198       0.807068 -0.030870
1    42        0.787625       0.805344 -0.017719
2   150        0.785181       0.806868 -0.021687
Summary:


{'baseline_mean': np.float64(0.7830013508598009),
 'baseline_std': np.float64(0.0060172102872899),
 'ablated_mean': np.float64(0.806426520148913),
 'ablated_std': np.float64(0.0009427132895643448),
 'delta_mean': np.float64(-0.02342516928911209),
 'delta_std': np.float64(0.006745465982711234)}